In [1]:
import numpy as np
import pandas as pd


def tratar_atipicos_semana7(df):
    """
    Tratamiento de atípicos (Semana 7) sobre un DataFrame ya limpio
    de la Semana 6 (sin duplicados, texto normalizado).

    Trata:
      - monto_compra: cap superior por IQR, preservando ventas corporativas legítimas
      - fecha_pedido: conversión a datetime
      - edad_cliente: valores centinela -> NaN, luego imputación por mediana
      - precio_unitario: corrección de signo, valores centinela -> NaN, imputación por mediana
      - unidades_vendidas: corrección de signo, valores centinela -> NaN, imputación por mediana

    Devuelve el DataFrame tratado (no modifica el original in-place).
    """
    df = df.copy()

    # --- monto_compra: límites IQR y ventas corporativas legítimas ---
    Q1 = df["monto_compra"].quantile(0.25)
    Q3 = df["monto_compra"].quantile(0.75)
    IQR = Q3 - Q1
    limite_superior = Q3 + 1.5 * IQR

    df["es_venta_corporativa"] = (
        (df["monto_compra"] > limite_superior)
        & (df["monto_compra"] >= 0)
        & (df["unidades_vendidas"].isin([1, 2, 3, 4, 5]))
        & (~df["precio_unitario"].isin([50000000, 999000000]))
    )

    df.loc[~df["es_venta_corporativa"], "monto_compra"] = (
        df.loc[~df["es_venta_corporativa"], "monto_compra"].clip(upper=limite_superior, lower=0)
    )

    # --- fecha_pedido: tipar como datetime ---
    df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])

    # --- edad_cliente: valores centinela -> NaN ---
    sentinelas_edad = [-5, 0, 150, 200]
    df.loc[df["edad_cliente"].isin(sentinelas_edad), "edad_cliente"] = np.nan

    # --- precio_unitario: corregir signo, valores centinela -> NaN ---
    df.loc[df["precio_unitario"] < 0, "precio_unitario"] = df.loc[df["precio_unitario"] < 0, "precio_unitario"].abs()
    sentinelas_precio = [50000000, 999000000]
    df.loc[df["precio_unitario"].isin(sentinelas_precio), "precio_unitario"] = np.nan

    # --- unidades_vendidas: corregir signo, valores centinela -> NaN ---
    df.loc[df["unidades_vendidas"] < 0, "unidades_vendidas"] = df.loc[df["unidades_vendidas"] < 0, "unidades_vendidas"].abs()
    sentinelas_uv = [0, 5000, 9999]
    df.loc[df["unidades_vendidas"].isin(sentinelas_uv), "unidades_vendidas"] = np.nan

    # --- Imputación por mediana con columnas de trazabilidad ---
    for columna in ["edad_cliente", "precio_unitario", "unidades_vendidas"]:
        mediana = df[columna].median()
        df[f"{columna}_era_nulo"] = df[columna].isna()
        df[columna] = df[columna].fillna(mediana)

    return df

In [2]:
df_s06 = pd.read_csv("../../data/S06_PD_Diaz_DatasetDepurado.csv")
df_s07 = tratar_atipicos_semana7(df_s06)

# Criterio: monto_compra ya no supera el límite superior, salvo ventas corporativas documentadas
Q1 = df_s06["monto_compra"].quantile(0.25)
Q3 = df_s06["monto_compra"].quantile(0.75)
limite_superior = Q3 + 1.5 * (Q3 - Q1)

excedidos_no_corporativos = df_s07[
    (df_s07["monto_compra"] > limite_superior) & (~df_s07["es_venta_corporativa"])
]
print("Filas que exceden el límite sin ser venta corporativa:", len(excedidos_no_corporativos))  # debe ser 0
print("Columna es_venta_corporativa presente:", "es_venta_corporativa" in df_s07.columns)  # debe ser True

Filas que exceden el límite sin ser venta corporativa: 0
Columna es_venta_corporativa presente: True
